In [ ]:
# Standard library
import json
from copy import copy
from math import ceil, isfinite
from pathlib import Path
from time import monotonic
from typing import Any

# Math
import numpy as np
from numpy import array, deg2rad, eye, mean, ndarray, pi, set_printoptions, stack, rad2deg, deg2rad
from numpy.random import seed, standard_normal
from scipy.stats import norm, uniform

# Data Handling
import utm
import pandas as pd
from filterpy.monte_carlo import systematic_resample, stratified_resample

# Other Third Party
from rich.progress import track

# Plotting
import matplotlib
from keplergl import KeplerGl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.transforms import Affine2D
from seaborn import move_legend
from svgpath2mpl import parse_path
from svgpathtools import svg2paths
import seaborn as sns

# ProMis
from promis import ProMis, DeltaStaRMap
from promis.geo import (
    CartesianCollection,
    DeltaGrid,
    CartesianDeltaCollection,
    CartesianLocation,
    CartesianMap,
    CartesianRasterBand,
    PolarCollection,
    PolarLocation,
)
from promis.loaders import OsmLoader
from promis.logic.spatial import Depth


In [ ]:
# This will auto-relead changed ProMis imports
%reload_ext autoreload
%autoreload 2

# The inD Dataset
https://levelxdata.com/ind-dataset/

https://levelxdata.com/wp-content/uploads/2024/03/inD-Format_1_1.pdf

- verschieben der Karte entsprechend der Umgebung
- anpassen der Weite
- für jede situation, (groupby frame?) berechne eine landscape, in der alle agenten verzeichnet sind.
- wir selbst stehen in der Mitte. Unser Heading / speed ist lowkey egal dafür? vielleicht auch hierzu nochmal mit / ohne experimente machen.

In [ ]:
# data_path = Path(".") / "data" / "athens_cars"
# car_data = pd.read_csv(data_path / "20181101_d9_0830_0900.csv", sep='; ', header=0)
data_path = Path(".") / "data" / "inD-dataset-v1.1" / "data"

inD_index = "08"
metadata = pd.read_csv(data_path / f"{inD_index}_recordingMeta.csv", sep=',')
car_data = pd.read_csv(data_path / f"{inD_index}_tracks.csv", sep=',')
car_metadata = pd.read_csv(data_path / f"{inD_index}_tracksMeta.csv", sep=',')
assert metadata["recordingId"].unique() == car_data["recordingId"].unique() == car_metadata["recordingId"].unique()
metadata

In [ ]:
# naive center of the scene, but not our origin of coordinates
# we use this as a bootstrap
origin = PolarLocation(
    latitude=metadata["latLocation"][0], longitude=metadata["lonLocation"][0]
)

# the dataset uses a local coordinate frame based on the UTM projection.
# the local frame resembles a CartesianCollection, we just need to transform its origin point
utm_number = utm.latlon_to_zone_number(origin.latitude, origin.longitude)
utm_letter = utm.latitude_to_zone_letter(origin.latitude)

origin_coords = utm.to_latlon(
    easting=metadata["xUtmOrigin"][0],
    northing=metadata["yUtmOrigin"][0],
    zone_number=utm_number,
    zone_letter=utm_letter,
)
origin = PolarLocation(longitude=origin_coords[1], latitude=origin_coords[0])

x_utm = car_data["xCenter"] + metadata["xUtmOrigin"][0]
y_utm = car_data["yCenter"] + metadata["yUtmOrigin"][0]
polar_car_coords = np.array(
    [utm.to_latlon(x, y, utm_number, utm_letter) for x,y in zip(x_utm, y_utm)]
)[:, ::-1] # polar coordinates are parsed longitude first, but the conversion returns coordinates in reverse order
polar_car_coord_collection = PolarCollection(origin)
polar_car_coord_collection.append_with_default(polar_car_coords, 0)
cartesian_car_cords = polar_car_coord_collection.to_cartesian().coordinates()

car_data["xCenter_utm"] = car_data["xCenter"]
car_data["yCenter_utm"] = car_data["yCenter"]
car_data["xCenter"] = cartesian_car_cords[:, 0] 
car_data["yCenter"] = cartesian_car_cords[:, 1] 

# we now want to translate the origin to the approximate center of the scene
# that way, we avoid to compute lots of uninteresting points
bounds = [[car_data[row].max(), car_data[row].min()] for row in ["xCenter", "yCenter"]]
offsets = [mean(bound_pair) for bound_pair in bounds]
diameter = 60 # max([bound_pair[0] - bound_pair[1] for bound_pair in bounds]) * 1.2

origin = CartesianLocation(*offsets).to_polar(origin)
dimensions = width, height = diameter, diameter

collection = CartesianDeltaCollection(origin=origin, number_of_values=0)
car_data["xCenter"] = car_data["xCenter"] - offsets[0] + 0.5  # TODO: analyze offset scientifically
car_data["yCenter"] = car_data["yCenter"] - offsets[1] + 0.5
car_data["speed"] = np.sqrt(np.pow(car_data["xVelocity"], 2) + np.pow(car_data["yVelocity"], 2)) * 3.6
car_data = car_data.join(car_metadata[["trackId", "class"]], on="trackId", how="left", rsuffix="r", validate="m:1")

del car_data["trackIdr"]

collection.append_with_default(car_data[["xCenter", "yCenter", "heading", "speed"]].values, value=())
car_data


In [ ]:
dt = 1 / metadata["frameRate"]
car_data[car_data["trackId"] < 8]

In [ ]:
setting = f"inD-crossing-{inD_index}"
output_folder = Path(".") / "ground-cofi-exports" / "naive" / setting
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
feature_description = {
    # "primary": "['highway' = 'primary']",
    "secondary": "['highway' = 'secondary']",
    "tertiary": "['highway' = 'tertiary']",
    # "service": "['highway' = 'service']",
    # "residential": "['highway' = 'residential']",
    # "crossing": "['footway' = 'crossing']",
    # "living_street": "[highway=living_street]",
    "unclassified": "['highway' = 'unclassified']",
    "signal": "['highway' = 'traffic_signals']",
}

covariance = {
    "secondary": 2 * np.eye(2),
    "tertiary": 2 * np.eye(2),
    "crossing": 1.2 * np.eye(2),
    "unclassified": 1.5 * np.eye(2),
}

general_logic = """
    unclassified_correct(X) :-
        over(X, unclassified),
        %C unclassified_side_correct(X),
        state_speed(X, S),
        maxspeed(X, unclassified, MS),
        MS >= S.

    tertiary_correct(X) :-
        over(X, tertiary),
        %C tertiary_side_correct(X),
        state_speed(X, S),
        maxspeed(X, tertiary, MS),
        MS >= S.
        

    % Definition of a valid mission
    landscape(X) :-
        tertiary_correct(X);
        unclassified_correct(X).
"""

In [ ]:
reload_uam = True
uam: CartesianMap
if reload_uam or not Path(output_folder / "uam.pkl").exists():
    uam = OsmLoader(
        origin=origin,
        dimensions=dimensions,
        feature_description=feature_description,
        polygonize_routes=True,
        timeout=15,
    ).to_cartesian_map()
else: 
    uam = CartesianMap.load(output_folder / "uam.pkl")


In [ ]:
uam.apply_covariance(covariance)
uam.save(output_folder / "uam.pkl")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7,7))
cars = car_data[car_data["class"] == "car"]
trucks = car_data[car_data["class"] == "truck_bus"]


for i, df in enumerate([cars]):
    for trackId in df["trackId"].unique():
        track = df[df["trackId"] == trackId]
        ax.plot(track["xCenter"], track["yCenter"], alpha = 0.3, color = "red")
        ax.plot(track["xCenter_utm"], track["yCenter_utm"], alpha = 0.3, color = "blue")


In [ ]:
reload_star = True
dsm: DeltaStaRMap
if reload_star:
    support = DeltaGrid(origin, (40, 40), width, height, speed_res=5, speed_bounds=(35, 55), bearing_res=1)
    dsm = DeltaStaRMap(uam)
    dsm.initialize(support, 15, general_logic)
    dsm.save(output_folder / "star.pkl")
else:
    dsm = DeltaStaRMap.load(output_folder / "star.pkl")

In [ ]:
print(dsm.relations)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12, 4))
params = dsm.get("over", "unclassified").parameters
params.number_of_values = 2
plot = params.scatter(value_index=0, bearing=235, speed=35, alpha=0.4, cmap="coolwarm_r", ax=ax[0])
plt.colorbar(plot)
params = dsm.get("over", "tertiary").parameters
plot = params.scatter(value_index=0, bearing=0, speed=30, alpha=0.4, cmap="coolwarm_r", ax=ax[1])
plt.colorbar(plot)
df = car_data[car_data["class"] == "car"]
for trackId in df["trackId"].unique():
    track = df[df["trackId"] == trackId]
    ax[1].plot(track["xCenter"], track["yCenter"], alpha = 0.4)

ax[0].set_title("P(over, unclassified)")
ax[1].set_title("P(over, tertiary), with given trajectories")
fig.suptitle("different StaRs on tertiary road type")

add an any location type, 
left_side, right_side? das könnte es eleganter machen

In [ ]:
ax = plt.subplot(xlim=(-30, 30), ylim=(-30, 30))
from promis.geo import CartesianPolygon
for f in uam.features:
    if f.location_type in ["tertiary", "unclassified"] and isinstance(f, CartesianPolygon): 
        ax.plot(*f.geometry.exterior.xy)
        print(f.tags)

In [ ]:
promis = ProMis(dsm)
promis.solve(support, general_logic, print_first=True, show_progress=True)
output_folder

In [ ]:
plot = support.scatter(bearing = 90, speed = 40, plot_basemap=True, alpha=0.5, cmap = "coolwarm_r")
plt.colorbar(plot)
for trackId in range(1):
    track = df[df["trackId"] == trackId]
    plt.plot(track["xCenter"], track["yCenter"], linewidth=4)
pass

In [ ]:
    
def experiment(
    vehicle_df: pd.DataFrame,
    seed_value: int = 2024,
    constitutional_trust: float = 0.8,
    interpolator: Any | None = None,
):
    def create_initial_particles(N: int) -> ndarray:
        return np.stack(
            [
                uniform(loc=-width / 2, scale=width).rvs(N),  # x position (east)
                uniform(loc=-height / 2, scale=height).rvs(N),  # y position (north)
                uniform(loc=0, scale=2 * pi).rvs(N),  # heading
                norm(loc=40, scale=10).rvs(N),  # speed
            ]
        ).T
    
    def predict(particles, dt, process_noise):
        """move according to control input u (heading change, velocity)
        with noise Q (std heading change, std velocity)`"""

        particles = particles.copy()
        N = len(particles)

        # We first add noise to the heading
        particles[:, 2] += deg2rad(
            45 * standard_normal(N) / dt  # we can turn 90 degrees per second in each direction (95% interval) # but this is per time step? = 1/25th sec
        )
        particles[:, 2] %= 2 * pi

        # Add noise to the speed
        particles[:, 3] += 20  * standard_normal(N) / dt # pm 40 kmh 

        # Compute the actual distance we travel
        distance = particles[:, 3] * dt / 3.6 # to scale to m/s
        particles[:, 0] += np.cos(particles[:, 2]) * distance
        particles[:, 1] += np.sin(particles[:, 2]) * distance

        return particles

        # scaled_process_noise = process_noise * dt

        # p = particles[:, [0, 1]]
        # v = particles[:, [2, 3]]

        # p_next = p + v * dt
        # v_next = v + np.random.normal(
        #     (0, 0), (scaled_process_noise, scaled_process_noise), size=(N, 2)
        # )

        # particles[:, [0, 1]] = p_next
        # particles[:, [2, 3]] = v_next

        return particles
    
    def update(
        particles,
        weights,
        z,
        R,
        dt,
        constitutional_trust: float = constitutional_trust,
    ):
        weights = weights.copy()

        positions = particles[:, :2]
        distance = np.linalg.norm(positions - z, axis=1)

        # Evaluate an RBF kernel
        if dt > 0.01:
            weights *= norm.pdf(distance, 0, R * dt)

        if constitutional_trust > 0:
            this_interpolator = interpolator
            constitution = this_interpolator(particles)[:, 0]
            weight_change = constitutional_trust * constitution + (1 - constitutional_trust)
            # Disallow going to placed where the constitution is not even defined
            weight_change[~np.isfinite(constitution)] = 0
            weights *= weight_change

        weights += 1.0e-300  # avoid round-off to zero
        weights /= sum(weights)  # normalize

        return weights
    

    def estimate(particles, weights):
        """returns mean and variance of the weighted particles"""

        mean = np.average(particles, weights=weights, axis=0)
        var = np.average((particles - mean) ** 2, weights=weights, axis=0)
        return mean, var

    def neff(weights):
        return 1.0 / np.sum(np.square(weights))

    def resample_from_index(particles, weights, indexes):
        particles[:] = particles[indexes]
        weights.resize(len(particles))
        weights.fill(1.0 / len(weights))

    def run_pf1(
        N: int,
        data: np.ndarray,
        sensor_std_err,
        process_noise,
        do_plot=False,
        plot_particles=True,
        plot_every=3,
    ):
        if do_plot:
            plt.figure()

        # Create particles uniformly over the entire space
        particles = create_initial_particles(N)

        # However, for the location we can do a bit better by assuming that the current
        # observation is approximately correct
        particles[:, 0] = norm(loc=data[0, 0], scale=10).rvs(N)
        particles[:, 1] = norm(loc=data[0, 1], scale=10).rvs(N)

        # create weights
        weights = np.ones(N) / N

        if do_plot and plot_particles:
            alpha = 0.20
            if N > 5000:
                alpha *= np.sqrt(5000) / np.sqrt(N)
            plt.scatter(particles[:, 0], particles[:, 1], alpha=alpha, color="g")
            plt.show()

        weights_trace = []
        particle_trace = []
        xs = []
        for i, obs in enumerate(data):
            dt = obs[4]

            # move
            particles = predict(particles, dt=dt, process_noise=process_noise)

            # incorporate measurements
            weights = update(particles, weights, z=obs[:2], R=sensor_std_err, dt=dt)

            # resample if too few effective particles
            if neff(weights) < N / 2:
                indexes = systematic_resample(weights)
                resample_from_index(particles, weights, indexes)
                assert np.allclose(weights, 1 / N)

            mu, var = estimate(particles, weights)
            xs.append(mu)

            weights_trace.append(weights.copy())
            particle_trace.append(particles.copy())

            if do_plot and i % plot_every == 0:
                if plot_particles:
                    plt.scatter(particles[:, 0], particles[:, 1], alpha=alpha, color="g")
                # draw a cricle perimeter at 1 std
                plt.gca().add_artist(
                    plt.Circle((mu[0], mu[1]), radius=sensor_std_err * dt, fill=False)
                )
                p1 = plt.scatter(obs[0], obs[1], marker="x", color="r")
                p2 = plt.scatter(mu[0], mu[1], marker="s", color="b")
                plt.xlim(-width / 2, width / 2)
                plt.ylim(-height / 2, height / 2)
                plt.show()

        xs = np.array(xs)
        if do_plot:
            plt.legend([p1, p2], ["Actual", "PF"], loc=4, numpoints=1)
            plt.show()

        pos_error = np.linalg.norm(xs[:, :2] - data[:, :2], axis=1)
        all_error = np.linalg.norm(xs[:, :] - data[:, :4], axis=1)

        if do_plot:
            plt.show()

        return {
            "particles": array(particle_trace),
            "weights": array(weights_trace),
            "estimates": xs,
            "pos_error": pos_error,
            "pos_error_mean": pos_error.mean(),
            "all_error": all_error,
            "all_error_mean": all_error.mean(),
            "truth": data,
        }


    ground_truth = stack(
        (
            vehicle_df["xCenter"],
            vehicle_df["yCenter"],
            deg2rad(vehicle_df["heading"]),
            vehicle_df["speed"],
            np.repeat(dt, len(vehicle_df))
        )
    ).T
    
    take_every = 5
    ground_truth = ground_truth[::take_every, :]

    seed(seed_value)
    return run_pf1(
        N=500,
        data=ground_truth,
        sensor_std_err=5,
        process_noise=0.02,
        # do_plot=True,
        # plot_every=25,
    )

trackId = 0
res_specific = experiment(
    df[df["trackId"] == trackId],
    seed_value=2025,
    constitutional_trust=0.5,
    interpolator=support.get_interpolator(),
)


In [ ]:
plot = support.scatter(bearing = 90, speed = 40, plot_basemap=True, alpha=0.5, cmap = "coolwarm_r")
plt.colorbar(plot)
track = df[df["trackId"] == trackId]
plt.plot(track["xCenter"], track["yCenter"], linewidth=4, color="black")
estimates = res_specific["estimates"]
plt.plot(estimates[:, 0], estimates[:, 1], linewidth=3, alpha=0.6, color="green")

In [ ]:
res_specific["pos_error_mean"]

Now that a single example runs through, it is time to evaluate the performance of a particle filter both with our constitution and without it. What split to use?